# Chapter 14 &mdash; Semi-Deciders, and Why RE Languages Are Procedures

**Concept 6 of the Chapter 14 decomposition:** *Semi-Deciders, and Why RE Languages Are Procedures*

A semi-decider halts and says "yes" only when the answer is yes; otherwise you wait, possibly forever.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Semi-Deciders/Concept-Semi-Deciders.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A **semi-decider** for $L$:

* on $w \in L$ it **halts and accepts**;
* on $w \notin L$ it may reject, or may **run forever**.

That is exactly "$L$ is RE". The machine implements a **procedure**, not an algorithm.

The practical shape of a semi-decider is a **search**: enumerate candidate witnesses
and stop when one works. If a witness exists you will find it; if none does, you search
forever.

And the crucial usage note: **a semi-decider's silence is not a 'no'.** Any system
built on one must have a story for "still running" that is not "therefore false".

## 2. Definitions

### A search-shaped semi-decider

In [ ]:
def semi_decide_has_factor(n, limit=None):
    # semi-decides "n is composite" by searching for a factor
    k, steps = 2, 0
    while limit is None or steps < limit:
        steps += 1
        if k * k > n: 
            if limit is None: return (False, steps)     # only terminates for small n
            return (None, steps)
        if n % k == 0: return (True, steps)
        k += 1
    return (None, steps)                                 # 'not yet'

### A TM semi-decider, for contrast with a decider

In [ ]:
Semi = md2mc('''TM
!! accepts tapes containing a 1; loops forever on 0^n
I : 1 ; 1 , R -> F
I : 0 ; 0 , R -> I
I : . ; . , R -> I
''')

# --- thin wrappers over Jove's TM runner --------------------------------
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

## 3. Tests

On a member, the semi-decider halts and says yes.

In [ ]:
for n in [9, 15, 91]:
    ans, steps = semi_decide_has_factor(n, limit=50)
    print("  %-4d composite? %-6s after %d steps" % (n, ans, steps))
assert semi_decide_has_factor(9, limit=50)[0]

On a non-member, you get 'not yet' &mdash; for as long as you care to wait.

In [ ]:
for limit in [3, 10, 40]:
    ans, steps = semi_decide_has_factor(101, limit=limit)
    print("  limit %2d : 101 composite? %-6s after %d steps" % (limit, ans, steps))
print("\nEvery answer is None.  None is not False.")

The TM version behaves the same way.

In [ ]:
print("'001' contains a 1 :", tm_halts(Semi, '001', fuel=100),
      " accepted :", tm_accepts(Semi, '001', fuel=100))
print("'000' contains no 1:", tm_halts(Semi, '000', fuel=500))
assert tm_accepts(Semi, '001', fuel=100)
assert not tm_halts(Semi, '000', fuel=500)

**Silence is not a 'no'.** This is the mistake to design against.

In [ ]:
print("observed: the machine has not answered")
print("possible: the answer is 'no'")
print("possible: the answer is 'yes' and it needs one more step")
print()
print("A system that treats 'no answer yet' as 'no' is unsound.")
print("A system that waits is unresponsive.  Pick your poison deliberately.")

Search is the canonical semi-decider shape.

In [ ]:
SHAPES = [("is n composite",        "search for a factor"),
          ("does M accept w",       "run M on w"),
          ("is this formula SAT",   "search for an assignment"),
          ("do G1 and G2 differ",   "enumerate strings in numeric order (Concept 8)")]
for q, how in SHAPES: print("  %-24s %s" % (q, how))
print("\nEach halts when it FINDS something.  None halts when there is nothing.")

## 4. Exercises


1. Turn the composite semi-decider into a decider. What did you have to know?
2. Which of the four searches above can be turned into deciders?
3. Why is "run $M$ on $w$" a semi-decider for $A_{TM}$ and not a decider?

In [ ]:
# Your work for the exercises above.